In [ ]:
# set device for inference
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model_rw2 = model_rw2.to(device)
model_rw2.eval()

In [ ]:
# get web frontend class
frontend_class = le.transform(["web_frontend"])[0]

In [ ]:
# import captum integrated gradients
from captum.attr import LayerIntegratedGradients
import torch
import re

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model_rw2 = model_rw2.to(device)
model_rw2.eval()

def forward_func(input_ids, attention_mask):
    return model_rw2(input_ids=input_ids, attention_mask=attention_mask).logits


def ig_words_for_text(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=50000
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    lig = LayerIntegratedGradients(
        forward_func,
        model_rw2.bert.embeddings
    )

    attributions = lig.attribute(
        inputs=input_ids,
        additional_forward_args=(attention_mask,),
        target=int(frontend_class),
        n_steps=6
    )

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu())

    scores = (
        attributions.sum(dim=-1)
        .squeeze()
        .detach()
        .cpu()
        .numpy()
    )

    token_scores = list(zip(tokens, scores))

    return token_scores

In [ ]:
# interpretability tokens attribution utility

def merge_wordpieces(tokens, scores):

    words = []
    current_word = ""
    current_score = 0

    for tok, score in zip(tokens, scores):

        if tok.startswith("##"):
            current_word += tok[2:]
            current_score += score
        else:
            if current_word:
                words.append((current_word, current_score))
            current_word = tok
            current_score = score

    if current_word:
        words.append((current_word, current_score))

    return words

In [ ]:
# attribution top words extraction

def clean_tokens(word_scores):

    cleaned = []

    for word, score in word_scores:

        word = word.lower()

        if word in ["[cls]", "[sep]"]:
            continue

        if len(word) < 3:
            continue

        if word.isdigit():
            continue

        cleaned.append((word, score))

    return cleaned

correct_front = df_test[
    (y_true_rw2 == frontend_class) &
    (y_pred_rw2 == frontend_class)
]

text = correct_front.iloc[0]["resume_text"]

In [ ]:
# extract tokens for attribution
tokens = ig_words_for_text(text)

cleaned = clean_tokens(tokens)

top_words = merge_wordpieces(tokens, scores)
top_words = sorted(cleaned, key=lambda x: abs(x[1]), reverse=True)[:10]

top_words

In [ ]:
# plot attribution bar chart
import matplotlib.pyplot as plt

words = [w for w,_ in top_words]
scores = [s for _,s in top_words]

colors = ["green" if s > 0 else "red" for s in scores]

plt.figure(figsize=(8,5))
plt.barh(words, scores, color=colors)

plt.xlabel("Integrated Gradients attribution")
plt.title("Important tokens for web_frontend prediction")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()